# Introduction to Machine Learning
## Southampton Statistics Summer School | 30 July 2026
### Brody Hannan, University of Southampton

---

**GitHub:** https://github.com/BHannan01/introduction-to-machine-learning
**Contact:** brody.hannan@soton.ac.uk

## How to Use This Notebook

This notebook runs in **Google Colab** — a free, browser-based Python environment. No installation required.

**To run a code cell:** click on it and press `Shift+Enter`, or click the ▶ button on the left.

**Work through the cells in order**, top to bottom. Each section builds on the previous one.

Look out for **🔍 Reflection Questions** throughout. These ask you to *interpret* the output, not just run the code. Spend a minute thinking before moving on.

---

## Learning Objectives

By the end of this practical, you will be able to:

1. Load and explore a real social science dataset in Python
2. Preprocess data for machine learning (missing values via multiple imputation, encoding, splitting)
3. Build and evaluate a **decision tree** classifier, including tuning the depth parameter
4. Build and evaluate a **random forest** classifier and interpret variable importance
5. Build and evaluate a **gradient boosting** classifier
6. Compare model performance using accuracy, precision, recall, and F1
7. Write up results in APA format
8. *(Extension)* Apply **k-means clustering** to discover latent groups

---
## Section 0: Setup

Run this cell first. It imports all packages needed for the workshop.

In [ ]:
# ── Core data packages ────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Machine learning (scikit-learn) ──────────────────────────────────────────
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# Enable the iterative imputer (experimental but stable in practice)
from sklearn.experimental import enable_iterative_imputer  # noqa — must be imported before IterativeImputer
from sklearn.impute import IterativeImputer

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print("All packages loaded successfully.")

---
## Section 1: The Dataset

We are using the **1994 US Census Income dataset** (also known as the "Adult" dataset).

**Source:** Becker, B. & Kohavi, R. (1996). *Adult*. UCI Machine Learning Repository.
https://archive.ics.uci.edu/dataset/2/adult

### The Task

Predict whether a person's annual income exceeds **\$50,000**, based on demographic and employment information.

This is a **binary classification** problem:
- **Outcome:** `>50K` or `<=50K`
- **Features:** demographics, education, employment characteristics

In [ ]:
print("Downloading dataset... (this may take 20-30 seconds on first run)")

data = fetch_openml('adult', version=2, as_frame=True, parser='auto')

df = data.data.copy()
df['income'] = data.target

print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
# First five rows
df.head()

In [ ]:
# Column names, data types, and non-null counts
df.info()

---
## Section 1b: What Do These Variables Actually Mean?

Before modelling, you must understand what each variable represents.
This is a **data dictionary** — something you should always build for a new dataset.

| Variable | Type | Description | Categories / Range |
|---|---|---|---|
| `age` | Continuous | Age in years | 17–90 |
| `workclass` | Categorical | Type of employer | Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked |
| `fnlwgt` | Continuous | **Census sampling weight** — how many people in the US population this row represents | Large integers |
| `education` | Categorical | Highest level of education attained | Preschool, 1st-4th, 5th-6th, 7th-8th, 9th, 10th, 11th, 12th, HS-grad, Some-college, Assoc-voc, Assoc-acdm, Bachelors, Masters, Prof-school, Doctorate |
| `education-num` | Continuous | Numeric encoding of `education` (1 = Preschool, 16 = Doctorate) | 1–16 |
| `marital-status` | Categorical | Marital status | Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse |
| `occupation` | Categorical | Broad job category | Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces |
| `relationship` | Categorical | Relationship to household head | Husband, Wife, Own-child, Not-in-family, Other-relative, Unmarried |
| `race` | Categorical | Race/ethnicity | White, Black, Asian-Pac-Islander, Amer-Indian-Eskimo, Other |
| `sex` | Categorical | Sex | Male, Female |
| `capital-gain` | Continuous | Investment income (dollars) this year | 0–99,999 |
| `capital-loss` | Continuous | Investment losses (dollars) this year | 0–4,356 |
| `hours-per-week` | Continuous | Average hours worked per week | 1–99 |
| `native-country` | Categorical | Country of origin | 41 countries |
| `income` | **Outcome** | Annual income above/below \$50K | \>50K, <=50K |

> **Important note on `fnlwgt`:** This is a **survey design weight**, not a predictor of income. It tells the census how many US citizens each sampled person represents. We will **exclude** it from the model. Including sampling weights as features is a common mistake that can distort results.

> **Note on `education` vs `education-num`:** These are the same variable in two forms — one categorical (word labels) and one numeric (1–16 scale). We use `education-num` (the numeric version) and drop the categorical `education` to avoid redundancy.

In [ ]:
# ── Inspect the unique categories in each categorical column ─────────────────
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

print("Categories in each categorical variable:")
print("=" * 60)
for col in categorical_cols:
    cats = sorted(df[col].dropna().unique().astype(str))
    print(f"\n{col} ({len(cats)} categories):")
    print("  " + ", ".join(cats))

In [ ]:
# ── Descriptive statistics for numeric variables ──────────────────────────────
# .describe() gives count, mean, std, min, quartiles, max for each numeric column.
df.describe().round(1)

---
## Section 2: Missing Values and Multiple Imputation

### Step 1: Identify missing values

In [ ]:
# The dataset uses '?' as a placeholder for unknown values.
# We replace these with NaN (Python's standard missing value marker) to detect them.

df_work = df.replace('?', float('nan'))

missing = df_work.isnull().sum()
missing_pct = (missing / len(df_work) * 100).round(1)

print("Missing values by column:")
print("-" * 40)
has_missing = missing[missing > 0]
for col in has_missing.index:
    print(f"  {col:<22}  {missing[col]:>5} missing  ({missing_pct[col]}%)")

print(f"\nTotal rows in dataset: {len(df_work):,}")
print(f"Columns with missing data: {len(has_missing)}")

In [ ]:
# Visualise the missing data pattern
fig, ax = plt.subplots(figsize=(8, 4))

missing_plot = missing_pct[missing_pct > 0].sort_values(ascending=True)
missing_plot.plot(kind='barh', ax=ax, color='#003B5C', edgecolor='white')
ax.set_title('Percentage Missing by Variable', fontsize=12, fontweight='bold')
ax.set_xlabel('% Missing')
ax.axvline(x=5, color='#C4A000', linestyle='--', linewidth=1.5, label='5% threshold')
ax.legend()
plt.tight_layout()
plt.show()

print("Note: missing values appear in workclass, occupation, and native-country only.")

### Step 2: Understanding Why Data is Missing

Before imputing, good research practice asks *why* values are missing.
There are three missing data mechanisms:

| Mechanism | Definition | Implication |
|---|---|---|
| **MCAR** (Missing Completely At Random) | Missingness is unrelated to any data | Safe to impute; complete-case analysis also valid |
| **MAR** (Missing At Random) | Missingness depends on *observed* variables, not on the missing value itself | Multiple imputation (MI) works well |
| **MNAR** (Missing Not At Random) | Missingness depends on the value that is missing | Most problematic; MI may be biased |

**Little's MCAR test** (Little, 1988) formally tests H₀: data is MCAR. In R, `mice::TestMCARNormality()` implements this. In Python, it is not in scikit-learn, but the `missingpy` and `pyampute` packages provide implementations.

For this dataset, the missing values in `workclass` and `occupation` are most likely **MAR** — people who did not report their occupation were probably not working. We proceed with multiple imputation, which is valid under both MCAR and MAR.

> **In your own research:** always run Little's MCAR test and inspect patterns with a missing data heatmap before imputing. Report how you handled missing data in your methods section.

### Step 3: Multiple Imputation

Rather than dropping rows with missing values (which loses data and can introduce bias), we use **iterative multiple imputation**, similar to the MICE algorithm widely used in social science statistics.

**How it works:**
1. For each variable with missing values, fit a predictive model using all other variables
2. Predict the missing values from that model
3. Repeat across all variables with missing data, cycling multiple times until predictions stabilise

This is implemented in scikit-learn as `IterativeImputer`, which is equivalent to what R's `mice` package calls *predictive mean matching* for continuous variables.

In [ ]:
# ── Multiple imputation using IterativeImputer ────────────────────────────────
#
# IterativeImputer performs MICE-style iterative imputation:
# it models each incomplete variable as a function of all other variables,
# then cycles through until convergence.
#
# Since the missing columns (workclass, occupation, native-country) are categorical,
# we first encode them as ordinal numbers, impute, then round back to integers
# and decode to the original category labels.

# 1. Select only the columns we will use in the model (+ the three with missing values)
impute_cols = ['workclass', 'occupation', 'native-country',
               'age', 'education-num', 'hours-per-week',
               'capital-gain', 'capital-loss', 'marital-status',
               'relationship', 'race', 'sex']

df_imp = df_work[impute_cols].copy()

# 2. Identify which columns are categorical
cat_imp_cols  = [c for c in impute_cols if df_imp[c].dtype in ['object', 'category']]
num_imp_cols  = [c for c in impute_cols if c not in cat_imp_cols]

# 3. Ordinal-encode categoricals so IterativeImputer can use them
#    encoded_missing_value=np.nan tells the encoder to leave NaN as NaN (for imputation)
enc = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1,
    encoded_missing_value=np.nan
)
df_imp[cat_imp_cols] = enc.fit_transform(df_imp[cat_imp_cols])

# 4. Run iterative imputation
#    max_iter=10 — cycle through all variables 10 times
#    random_state=42 — reproducible result
imp = IterativeImputer(max_iter=10, random_state=42)
df_imp_array = imp.fit_transform(df_imp)
df_imp = pd.DataFrame(df_imp_array, columns=impute_cols)

# 5. Round categorical columns back to whole numbers and decode to text
for c in cat_imp_cols:
    df_imp[c] = df_imp[c].round().clip(0).astype(int)
df_imp[cat_imp_cols] = enc.inverse_transform(df_imp[cat_imp_cols])

# 6. Add back the outcome variable (income — no missing values)
df_imp['income'] = df_work['income'].values

# Check: no more missing values
remaining_missing = df_imp.isnull().sum().sum()
print(f"Missing values after imputation: {remaining_missing}")
print(f"Rows retained: {len(df_imp):,}  (all {len(df_work):,} rows kept)")

### 🔍 Reflection Question 1

- We retained all 48,842 rows after imputation, compared to losing about 3,000 rows if we had dropped missing values. Why does retaining more data matter for model training?
- The three variables with missing values (workclass, occupation, native-country) are all categorical. In what scenario might "missingness" in occupation be **informative** (i.e., tell us something about the person's income)?
- We imputed using a MICE-style algorithm. In your own research, would you need to justify this choice in a methods section? What would you say?

---
## Section 3: Exploring the Data

Before modelling, always visualise your key variables.

In [ ]:
# ── Distribution of the outcome variable ─────────────────────────────────────
# Clean up any whitespace
df_imp['income'] = df_imp['income'].str.strip()

income_counts = df_imp['income'].value_counts()
income_pct    = df_imp['income'].value_counts(normalize=True) * 100

print("Income class distribution:")
print("-" * 38)
for label in income_counts.index:
    bar = '█' * int(income_pct[label] / 2)
    print(f"  {label:<8}  {income_counts[label]:>6,}  ({income_pct[label]:.1f}%)  {bar}")

### Understanding Class Imbalance

The outcome variable is **imbalanced**: about **75% of people earn ≤\$50K** and only **25% earn >\$50K**.

This matters because:

1. **Accuracy is misleading.** A model that predicts "≤$50K" for *every single person* — without ever looking at any features — would achieve **~75% accuracy**. That looks impressive but is completely useless. Always check what the *baseline accuracy* (majority class proportion) is before interpreting your model's accuracy score.

2. **How does training work with imbalanced data?** We do NOT take equal numbers of each class. We use `stratify=y` in our train/test split, which means both the training set and test set maintain the **same 75/25 split** as the original data. The model learns from the natural distribution. If we wanted to improve recall for the minority class (>\$50K), we could use `class_weight='balanced'` or oversampling techniques like SMOTE — but for this workshop we use the natural distribution.

3. **Is any data "lost" in the train/test split?** No. The test set (20% of rows) is held back *temporarily* for evaluation — it is not deleted. The training set (80%) is what the model learns from. The test set is only used at the very end to give an honest estimate of how the model would perform on new data it has never seen. Cross-validation is an alternative that uses *all* data for training in different folds.

**Our baseline to beat: ~75% accuracy (the majority-class classifier)**

In [ ]:
# ── Visualise key features ────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Age by income
df_imp.boxplot(column='age', by='income', ax=axes[0, 0],
               boxprops=dict(color='#003B5C'),
               medianprops=dict(color='#C4A000', linewidth=2.5),
               whiskerprops=dict(color='#003B5C'),
               capprops=dict(color='#003B5C'),
               flierprops=dict(marker='o', color='#003B5C', alpha=0.3, markersize=3))
axes[0, 0].set_title('Age by Income Group', fontweight='bold')
axes[0, 0].set_xlabel('')

# Education level by income
edu_income = df_imp.groupby(['education-num', 'income']).size().unstack(fill_value=0)
edu_income.plot(kind='bar', ax=axes[0, 1], color=['#003B5C', '#C4A000'],
                edgecolor='white', width=0.7)
axes[0, 1].set_title('Education Level by Income', fontweight='bold')
axes[0, 1].set_xlabel('Years of education')
axes[0, 1].legend(title='Income', fontsize=8)
axes[0, 1].tick_params(axis='x', rotation=0)

# Hours per week by income
df_imp.boxplot(column='hours-per-week', by='income', ax=axes[1, 0],
               boxprops=dict(color='#003B5C'),
               medianprops=dict(color='#C4A000', linewidth=2.5),
               whiskerprops=dict(color='#003B5C'),
               capprops=dict(color='#003B5C'),
               flierprops=dict(marker='o', color='#003B5C', alpha=0.3, markersize=3))
axes[1, 0].set_title('Hours/Week by Income Group', fontweight='bold')
axes[1, 0].set_xlabel('')

# Sex and income (as %)
sex_income = df_imp.groupby(['sex', 'income']).size().unstack(fill_value=0)
sex_income_pct = sex_income.div(sex_income.sum(axis=1), axis=0) * 100
sex_income_pct.plot(kind='bar', ax=axes[1, 1], color=['#003B5C', '#C4A000'],
                    edgecolor='white', width=0.55)
axes[1, 1].set_title('Income Distribution by Sex (%)', fontweight='bold')
axes[1, 1].set_ylabel('%')
axes[1, 1].legend(title='Income', fontsize=8)
axes[1, 1].tick_params(axis='x', rotation=0)

fig.suptitle('Exploratory Data Analysis', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 🔍 Reflection Question 2

- Which variables show the strongest relationship with income?
- The sex plot shows a striking disparity. Is `sex` a *cause* of income differences, or a proxy for other factors (e.g., occupational segregation, career breaks)?
- What are the ethical implications of including demographic variables like sex and race in a model that might be used to allocate resources?

---
## Section 4: Preprocessing

### Step 1: Encode the Outcome Variable

In [ ]:
# Convert '>50K' to 1 and '<=50K' to 0
# ML models need numbers, not text labels.
df_imp['income_binary'] = (df_imp['income'] == '>50K').astype(int)

print("Encoding check:")
print(df_imp[['income', 'income_binary']].value_counts().sort_index())

### Step 2: Feature Selection and Dummy Coding

We now convert the predictor variables into a format suitable for modelling.

**What is one-hot (dummy) coding?**

Social scientists often call this **dummy variable coding** — it's the same concept.
For a categorical variable like `sex` (Male/Female), we create one binary column: `sex_Male` = 1 if Male, 0 if Female.
For a variable with more categories (like `occupation` with 14 categories), we create 13 binary columns (one is dropped to avoid perfect multicollinearity — the "reference category").

This is identical to what SPSS, Stata, and R do when you enter categorical predictors into a regression model.

`pd.get_dummies(drop_first=True)` automates this for every text/category column in the dataframe.

**Note on `fnlwgt`:** We exclude this variable — it is a census sampling weight, not a substantive predictor of income.

In [ ]:
# ── Select features ───────────────────────────────────────────────────────────
feature_cols = [
    'age',              # Continuous
    'education-num',    # Continuous (numeric education level, 1-16)
    'hours-per-week',   # Continuous
    'capital-gain',     # Continuous (investment income)
    'capital-loss',     # Continuous (investment losses)
    'workclass',        # Categorical (8 types of employer)
    'marital-status',   # Categorical (7 categories)
    'occupation',       # Categorical (14 job types)
    'relationship',     # Categorical (6 household roles)
    'race',             # Categorical (5 groups)
    'sex'               # Categorical (Male/Female)
    # Excluded: fnlwgt (sampling weight, not a predictor)
    # Excluded: education (redundant with education-num)
    # Excluded: native-country (very many small categories; adds noise)
]

X_raw = df_imp[feature_cols].copy()
y     = df_imp['income_binary'].copy()

# One-hot encode all categorical (text) columns
# drop_first=True creates k-1 dummies per variable (drops the reference category)
X = pd.get_dummies(X_raw, drop_first=True)

print(f"Feature matrix: {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"\nOriginal feature columns:  {len(feature_cols)}")
print(f"After dummy coding:         {X.shape[1]} columns")
print(f"\nFirst 6 column names:")
for c in list(X.columns[:6]):
    print(f"  {c}")
print("  ...")

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
# 80% training, 20% test.
# stratify=y preserves the 75/25 class ratio in BOTH sets.
# random_state=42 gives the same split every time you run this.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training set:  {X_train.shape[0]:,} rows")
print(f"Test set:      {X_test.shape[0]:,} rows")
print(f"\nClass balance check (stratify=y should keep both at ~75/25):")
print(f"  Training — % earning >$50K: {y_train.mean()*100:.1f}%")
print(f"  Test     — % earning >$50K: {y_test.mean()*100:.1f}%")
print(f"\nBaseline accuracy (always predict <=50K): {(1 - y.mean())*100:.1f}%")
print("  This is the floor we need to beat.")

---
## Section 5: Decision Tree

A decision tree learns a sequence of yes/no rules from the training data. Think of it as an automated flowchart the algorithm builds by learning from the data.

**How it works:**
1. Start with all training data at the root node
2. Try every feature and every possible threshold; choose the split that best separates the income classes
3. Splitting criterion: **Gini impurity** — measures how "mixed" the classes are (0 = perfectly pure; 0.5 = maximally mixed for binary outcomes)
4. Recurse on each branch until a stopping rule is reached

**Key parameter:** `max_depth` controls how many levels of splits the tree is allowed to make. A shallow tree underfits; a deep tree overfits by memorising the training data. We will experiment with different depths to find the best value.

### Step 1: Find the Best Depth Parameter

In [ ]:
# ── Depth exploration: find where test accuracy peaks ─────────────────────────
# We train a decision tree at each depth (1 to 15) and record:
#   - Training accuracy (how well it fits the training data)
#   - Test accuracy (how well it generalises to new data)
#
# A growing gap between the two is the hallmark of overfitting.

depths      = range(1, 16)
train_accs  = []
test_accs   = []

for d in depths:
    dt_temp = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt_temp.fit(X_train, y_train)
    train_accs.append(accuracy_score(y_train, dt_temp.predict(X_train)))
    test_accs.append(accuracy_score(y_test,  dt_temp.predict(X_test)))

# Plot the learning curves
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(depths, [a*100 for a in train_accs], 'o-', color='#003B5C',
        linewidth=2, markersize=6, label='Training accuracy')
ax.plot(depths, [a*100 for a in test_accs],  's-', color='#C4A000',
        linewidth=2, markersize=6, label='Test accuracy')

# Highlight the best test accuracy
best_depth = depths[int(np.argmax(test_accs))]
best_acc   = max(test_accs) * 100
ax.axvline(x=best_depth, color='red', linestyle='--', linewidth=1.5,
           label=f'Best depth = {best_depth}  ({best_acc:.1f}% test acc)')

ax.axhline(y=(1 - y.mean())*100, color='grey', linestyle=':', linewidth=1.5,
           label=f'Baseline ({(1-y.mean())*100:.1f}% — always predict <=50K)')

ax.set_title('Decision Tree: Training vs Test Accuracy by Depth',
             fontsize=13, fontweight='bold')
ax.set_xlabel('max_depth', fontsize=11)
ax.set_ylabel('Accuracy (%)', fontsize=11)
ax.set_ylim(70, 100)
ax.legend(fontsize=9)
ax.yaxis.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print(f"Best depth on test set: {best_depth}  (accuracy: {best_acc:.2f}%)")
print(f"Training accuracy at this depth: {train_accs[best_depth-1]*100:.2f}%")
print(f"Gap (overfit signal): {(train_accs[best_depth-1] - test_accs[best_depth-1])*100:.2f} percentage points")

### 🔍 Reflection Question 3

Look at the accuracy curves.

- At what depth does the training accuracy reach nearly 100%? What does that tell you?
- At what depth does the test accuracy stop improving (or start to drop)? This is the *sweet spot* — the point where the model is complex enough to learn the real patterns but not so complex that it memorises noise.
- What is the gap between training and test accuracy at depth 15? What does this gap represent in terms of overfitting?
- The grey line is the **baseline**: accuracy from predicting ≤$50K for everyone. At what depth does the decision tree first beat this baseline?

In [ ]:
# ── Build the final decision tree at the best depth ──────────────────────────
# We use max_depth=5 (slightly conservative — good balance of accuracy and interpretability).
# You can change this to best_depth if you prefer.

dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=30,   # Each leaf must contain at least 30 training observations
    random_state=42
)

dt.fit(X_train, y_train)

print(f"Decision tree trained at max_depth = {dt.max_depth}")
print(f"  Actual depth reached: {dt.get_depth()}")
print(f"  Number of leaves:     {dt.get_n_leaves()}")

### Step 2: Visualise the Tree

The plot below shows the full tree structure. Each box (node) contains four pieces of information:

| Item | Meaning |
|---|---|
| **Condition** (top line) | The yes/no question: e.g., `capital-gain <= 5095.5` |
| **gini** | Gini impurity at this node (0 = pure; 0.5 = maximally mixed for binary outcome). Lower is better. |
| **samples** | Number of training observations that reach this node |
| **value** | [count of <=50K, count of >50K] at this node |
| **class** | The majority class at this node — this is what the model would predict if a prediction were made here |

The **colour** indicates majority class: blue = predicts <=50K, orange = predicts >50K. Darker colour = more confident prediction (lower Gini).

In [ ]:
fig, ax = plt.subplots(figsize=(22, 10))

plot_tree(
    dt,
    feature_names=X.columns.tolist(),
    class_names=['<=50K', '>50K'],
    filled=True,
    rounded=True,
    fontsize=8,
    ax=ax
)

ax.set_title(f'Decision Tree Classifier (max_depth = {dt.max_depth})',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Tip: zoom into the image to read individual node details.")
print("Trace a path from the root to a leaf to see the rules for one prediction.")

### Step 3: Evaluate the Decision Tree

**What counts as a "good" accuracy?**

Context determines this, but here is a useful benchmark framework:

| Accuracy | Interpretation |
|---|---|
| Below baseline (~75%) | Worse than doing nothing — something is wrong |
| 75–80% | Marginal improvement over baseline |
| 80–87% | Solid performance for this dataset; consistent with published literature |
| 87–90% | Strong performance; in line with best published results for this dataset |
| Above 90% | Check for data leakage (e.g., a feature that directly encodes the outcome) |

For the Adult Census dataset, published benchmark results typically range from **85–87%** accuracy for tree-based methods (Kohavi, 1996; Fernandes et al., 2024). Our results should fall in that range.

In [ ]:
# ── Evaluate on the held-out test set ────────────────────────────────────────
y_pred_dt = dt.predict(X_test)

acc_dt = accuracy_score(y_test, y_pred_dt)
baseline = (1 - y_test.mean())

print(f"Baseline accuracy (always predict <=50K): {baseline*100:.1f}%")
print(f"Decision Tree accuracy:                   {acc_dt*100:.1f}%")
print(f"Improvement over baseline:                {(acc_dt - baseline)*100:.1f} percentage points")
print()
print("Classification Report")
print("=" * 58)
print(classification_report(y_test, y_pred_dt, target_names=['<=50K', '>50K']))

### Reading the Classification Report

The classification report gives four metrics for each class:

| Metric | Formula | What it asks |
|---|---|---|
| **Precision** | TP / (TP + FP) | Of everyone the model *predicted* as >\$50K, what fraction actually earns >\$50K? (quality of positive predictions) |
| **Recall** | TP / (TP + FN) | Of all people who *actually* earn >\$50K, what fraction did the model correctly identify? (coverage of positives) |
| **F1-score** | 2 × (P × R) / (P + R) | Harmonic mean of precision and recall. Best when both matter equally. |
| **Support** | — | Number of actual observations in this class in the test set |

The **macro avg** averages across classes equally. The **weighted avg** weights by class size.

> **Key insight:** precision and recall for the >\$50K class are both lower than for <=\$50K. This is the effect of class imbalance — the model has seen 3× more examples of <=\$50K during training.

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
cm_dt = confusion_matrix(y_test, y_pred_dt)
ConfusionMatrixDisplay(cm_dt, display_labels=['<=50K', '>50K']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Decision Tree: Confusion Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm_dt.ravel()
total = len(y_test)
print(f"True Positives  — correctly identified >$50K:    {tp:>5,}  ({tp/total*100:.1f}%)")
print(f"True Negatives  — correctly identified <=$50K:   {tn:>5,}  ({tn/total*100:.1f}%)")
print(f"False Positives — predicted >$50K, wrong:        {fp:>5,}  ({fp/total*100:.1f}%)")
print(f"False Negatives — missed actual >$50K earners:   {fn:>5,}  ({fn/total*100:.1f}%)")
print()
print(f"Overall accuracy: {(tp+tn)/total*100:.1f}%")

### Is This Result Publishable?

Whether results are publishable depends on several things beyond the confusion matrix:

1. **Comparison to a baseline and to published work.** Our accuracy (~84–86%) is in line with published results for this dataset (Kohavi, 1996). That is a good sign.

2. **The research question.** If you are using ML *exploratorily* (to see which features matter), the variable importance is the finding — not the raw accuracy. If you are deploying the model to make decisions about people, accuracy requirements are much higher.

3. **Reporting standards.** A publishable ML study in social science typically reports: accuracy, precision, recall, F1 (at minimum), plus confidence intervals from cross-validation, variable importance, and a description of how missing data was handled.

4. **The confusion matrix matters more than accuracy alone.** A journal reviewer will ask: of the high earners you missed (false negatives), who were they? Were your errors clustered in a particular demographic group? That is a fairness analysis — and increasingly expected in social science ML papers.

> **Bottom line:** these results are consistent with the published literature and would be presentable in a paper, provided you also report cross-validated metrics, discuss class imbalance, and contextualise against the literature.

### 🔍 Reflection Question 4

Look at the confusion matrix.

- How many actual >\$50K earners did the model fail to identify (false negatives)? In plain language, what does that mean?
- Which type of error is larger: false positives or false negatives? Is one type of error more harmful than the other in a real-world application of this model?
- The model's accuracy is around 84–86%. We calculated the baseline (~75%). What is the "real" improvement? Is it meaningful?
- If this model were used by a government to identify high earners for a tax audit programme, which metric would matter most: precision or recall?

---
## Section 6: Random Forest

### What is a Random Forest — and is it just picking the best tree?

A random forest is **not** simply selecting the single best decision tree. It builds **hundreds of different trees** and combines all of them.

Here is how diversity is introduced:

| Source of diversity | How |
|---|---|
| **Bootstrap sampling** | Each tree is trained on a random 63% sample of the training data (with replacement). No two trees see exactly the same data. |
| **Random feature selection** | At each split, only a random subset of features is considered (typically √p features, where p is the total number). No two trees use the same feature combinations at each node. |

Because the trees make different errors in different places, those errors **cancel out** when you take a majority vote. A single biased tree might always get a certain type of person wrong — but 200 diverse trees are unlikely to all make the same mistake.

**The final prediction:** each of the 200 trees casts a vote. The class with the most votes wins. All 200 trees contribute equally — no tree is selected as "best".

In [ ]:
# ── Build the random forest ───────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200,     # Build 200 trees
    max_features='sqrt',  # At each split, consider sqrt(n_features) candidates
    random_state=42,
    n_jobs=-1             # Use all CPU cores
)

print("Training random forest (200 trees)...")
rf.fit(X_train, y_train)
print(f"Training complete: {rf.n_estimators} trees built")

In [ ]:
y_pred_rf = rf.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest Accuracy: {acc_rf*100:.1f}%")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['<=50K', '>50K']))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
cm_rf = confusion_matrix(y_test, y_pred_rf)
ConfusionMatrixDisplay(cm_rf, display_labels=['<=50K', '>50K']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Random Forest: Confusion Matrix', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### Variable Importance: What is the Model Actually Using?

After training, the random forest can tell us which features drove its predictions.

**What is Mean Decrease in Impurity (MDI)?**

At every node in every tree, the algorithm chooses a feature to split on. At each of those splits, the Gini impurity decreases (the classes become purer in the child nodes). The **MDI importance** of a feature is the *total reduction in Gini impurity* caused by splits on that feature, summed across all trees and all nodes.

A high MDI score means: "this feature was responsible for a lot of the purification of the tree — it separated the classes well."

**Limitation:** MDI can favour features with many unique values (like continuous variables and variables with many categories), even if they are not truly more important. A more robust alternative is **permutation importance** (`sklearn.inspection.permutation_importance`), which directly measures accuracy drop when feature values are randomly shuffled. For a workshop, MDI gives us the right intuition.

**This is not the same as a regression coefficient.** Variable importance tells us which features the model *used most* — it does not tell us the *direction* of the effect or whether it is a cause. For directionality, partial dependence plots (PDPs) or SHAP values are used.

In [ ]:
importances = pd.Series(
    rf.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
importances.head(15).sort_values().plot(
    kind='barh', ax=ax, color='#003B5C', edgecolor='white')
ax.set_title('Random Forest: Top 15 Most Important Features (MDI)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity', fontsize=10)
plt.tight_layout()
plt.show()

print("Top 5 features by MDI importance:")
for feat, imp in importances.head(5).items():
    print(f"  {feat:<40} {imp:.4f}")

### 🔍 Reflection Question 5

- Which features matter most? Do these match what you expected from the EDA plots?
- `capital-gain` often ranks first. What does this tell you about the structure of income inequality in this dataset? (Hint: who has capital gains at all?)
- Variable importance tells you what the *model used*. Can you infer from this which features *cause* income differences? What additional methodology would you need?
- One-hot-encoded variables (like `marital-status_Never-married`) appear individually. How would you assess the importance of the original variable `marital-status` as a whole? (Hint: you could sum across all its dummy columns.)

---
## Section 7: Gradient Boosting

Gradient boosting builds trees **sequentially**. Each new tree focuses specifically on the errors the current ensemble made. The name comes from the fact that each tree moves down the gradient of the loss function — it is learning in the direction that most reduces the current error.

This contrasts with random forests, where all trees grow independently in parallel.

**Key hyperparameters and their effects:**
- `learning_rate`: how much each tree's prediction contributes. Smaller = more conservative, needs more trees.
- `n_estimators`: number of boosting rounds. More rounds = better fit, but risk of overfitting.
- `max_depth`: depth of each individual tree. Gradient boosting works best with *shallow* trees (depth 3–5).

In [ ]:
gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    random_state=42
)

print("Training gradient boosting (this takes ~30-60 seconds)...")
gb.fit(X_train, y_train)
print("Done.")

In [ ]:
y_pred_gb = gb.predict(X_test)
acc_gb = accuracy_score(y_test, y_pred_gb)

print(f"Gradient Boosting Accuracy: {acc_gb*100:.1f}%")
print()
print("Classification Report:")
print(classification_report(y_test, y_pred_gb, target_names=['<=50K', '>50K']))

---
## Section 8: Model Comparison

Now we place all three models side-by-side on the held-out test set.

**What do these metrics mean — in plain language?**

| Metric | Plain language | When it matters most |
|---|---|---|
| **Accuracy** | What % of all predictions were correct? | When classes are balanced and all errors cost equally |
| **Precision** (>\$50K) | Of everyone the model labelled as a high earner, what % actually was? | When a false alarm is costly (e.g., wrongly flagging someone for tax audit) |
| **Recall** (>\$50K) | Of all actual high earners, what % did the model catch? | When missing a true case is costly (e.g., missing a high earner who should pay more tax) |
| **F1** (>\$50K) | Balances precision and recall into one score (0–1, higher is better) | When both false alarms and missed cases matter |

In [ ]:
def get_metrics(y_true, y_pred, name):
    # Returns a dict of key classification metrics for one model
    return {
        'Model':            name,
        'Accuracy':         round(accuracy_score(y_true, y_pred), 3),
        'Precision (>50K)': round(precision_score(y_true, y_pred, zero_division=0), 3),
        'Recall (>50K)':    round(recall_score(y_true, y_pred, zero_division=0), 3),
        'F1 (>50K)':        round(f1_score(y_true, y_pred, zero_division=0), 3)
    }

results = pd.DataFrame([
    get_metrics(y_test, y_pred_dt, 'Decision Tree'),
    get_metrics(y_test, y_pred_rf, 'Random Forest'),
    get_metrics(y_test, y_pred_gb, 'Gradient Boosting'),
])

print("Model Comparison — Test Set Performance")
print("=" * 62)
print(results.to_string(index=False))
print()
print(f"Baseline (always predict <=50K): Accuracy = {(1-y_test.mean())*100:.1f}%  |  Recall (>50K) = 0.0%")

In [ ]:
metrics = ['Accuracy', 'Precision (>50K)', 'Recall (>50K)', 'F1 (>50K)']
x = np.arange(len(metrics))
w = 0.25

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w, results.iloc[0][metrics], w, label='Decision Tree',
            color='#C4A000', edgecolor='white')
b2 = ax.bar(x,     results.iloc[1][metrics], w, label='Random Forest',
            color='#003B5C', edgecolor='white')
b3 = ax.bar(x + w, results.iloc[2][metrics], w, label='Gradient Boosting',
            color='#5C8FA8', edgecolor='white')

ax.set_title('Model Comparison: Test Set Performance', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0.5, 1.02)
ax.legend(fontsize=10)
ax.yaxis.grid(True, alpha=0.4)
ax.set_axisbelow(True)

for bars in [b1, b2, b3]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.005,
                f'{bar.get_height():.2f}',
                ha='center', va='bottom', fontsize=7.5)
plt.tight_layout()
plt.show()

### 🔍 Reflection Question 6

- Which model performs best overall? Is the improvement from decision tree to gradient boosting large or marginal?
- For a policy application targeting high earners, would you prioritise precision or recall? Which model would you choose?
- All three models used the same features. Could you improve performance further? What new features might you engineer from the existing variables?
- Gradient boosting took noticeably longer to train. If your dataset were 10× larger, how would you balance accuracy against training time?

---
## Section 9: Extension — K-Means Clustering

So far we have done **supervised learning** — we gave the model a labelled outcome and asked it to predict. Now we try **unsupervised learning**, where there is no outcome variable.

**Goal:** find natural groupings in the census population purely from their characteristics.

This is the data-driven equivalent of **typology building** in social science. Rather than constructing categories by hand based on theory, we let the data reveal its own structure.

In [ ]:
# Features to cluster on (all continuous — consistent with k-means assumptions)
cluster_features = ['age', 'education-num', 'hours-per-week',
                    'capital-gain', 'capital-loss']

# Standardise: k-means uses Euclidean distance, so all features need the same scale.
# StandardScaler gives each feature mean=0 and std=1.
scaler    = StandardScaler()
X_cluster = scaler.fit_transform(df_imp[cluster_features])

print(f"Clustering matrix: {X_cluster.shape[0]:,} rows x {X_cluster.shape[1]} features")
print("All features standardised to mean=0, std=1")

In [ ]:
# ── Elbow method ──────────────────────────────────────────────────────────────
# We try k from 1 to 10 and record Within-Cluster Sum of Squares (WCSS).
# WCSS measures how tightly packed clusters are. Lower = tighter clusters.
# We look for the 'elbow': diminishing returns from adding more clusters.

wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, 11), wcss, 'o-', color='#003B5C', linewidth=2, markersize=7)
ax.set_title('Elbow Method: Choosing Number of Clusters', fontsize=12, fontweight='bold')
ax.set_xlabel('Number of clusters (k)', fontsize=11)
ax.set_ylabel('Within-cluster sum of squares', fontsize=11)
ax.axvline(x=4, color='#C4A000', linestyle='--', linewidth=1.8, label='k = 4')
ax.legend(fontsize=10)
ax.yaxis.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Fit k-means with k=4 ─────────────────────────────────────────────────────
k      = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)

df_clust = df_imp.copy()
df_clust['cluster'] = kmeans.fit_predict(X_cluster)

# Encode income for profiling
df_clust['income_binary'] = (df_clust['income'] == '>50K').astype(int)

print(f"K-means fitted with k={k}")
print("\nCluster sizes:")
for c, cnt in df_clust['cluster'].value_counts().sort_index().items():
    print(f"  Cluster {c}: {cnt:,} people  ({cnt/len(df_clust)*100:.1f}%)")

In [ ]:
# ── Cluster profiles ──────────────────────────────────────────────────────────
profile = df_clust.groupby('cluster')[cluster_features + ['income_binary']].mean().round(2)
profile.columns = ['Mean Age', 'Mean Educ (yrs)', 'Mean Hrs/Wk',
                   'Mean Cap. Gain', 'Mean Cap. Loss', '% earns >$50K']
profile['% earns >$50K'] = (profile['% earns >$50K'] * 100).round(1)

print("Cluster Profiles (mean values):")
print("=" * 75)
print(profile.to_string())

In [ ]:
# ── Visualise clusters: percentage distributions ──────────────────────────────
# We use PERCENTAGES within each cluster (not raw counts) so that small
# and large clusters are directly comparable on the same scale.

colors = ['#003B5C', '#C4A000', '#5C8FA8', '#8B4513']
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# 1. Age distribution by cluster — as %
bins_age = np.arange(15, 95, 5)
for c in range(k):
    cluster_ages = df_clust[df_clust['cluster'] == c]['age']
    counts, edges = np.histogram(cluster_ages, bins=bins_age)
    pct = counts / counts.sum() * 100
    axes[0].plot((edges[:-1] + edges[1:]) / 2, pct,
                 color=colors[c], linewidth=2, label=f'Cluster {c}', marker='o', markersize=4)
axes[0].set_title('Age Distribution (% within cluster)', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('% of cluster')
axes[0].legend(fontsize=8)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

# 2. Education level by cluster — as %
bins_edu = np.arange(0.5, 17.5, 1)
for c in range(k):
    cluster_edu = df_clust[df_clust['cluster'] == c]['education-num']
    counts, edges = np.histogram(cluster_edu, bins=bins_edu)
    pct = counts / counts.sum() * 100
    axes[1].plot((edges[:-1] + edges[1:]) / 2, pct,
                 color=colors[c], linewidth=2, label=f'Cluster {c}', marker='s', markersize=4)
axes[1].set_title('Education Level (% within cluster)', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Years of education')
axes[1].set_ylabel('% of cluster')
axes[1].legend(fontsize=8)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

# 3. % earning >$50K by cluster
pct_high = df_clust.groupby('cluster')['income_binary'].mean() * 100
bars = axes[2].bar(range(k), pct_high.values, color=colors[:k], edgecolor='white', width=0.6)
axes[2].set_title('% Earning >$50K by Cluster', fontsize=10, fontweight='bold')
axes[2].set_xlabel('Cluster')
axes[2].set_ylabel('%')
axes[2].set_xticks(range(k))
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter())
for bar, val in zip(bars, pct_high.values):
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.5,
                 f'{val:.0f}%', ha='center', va='bottom', fontsize=9)

fig.suptitle('K-Means Cluster Profiles (k=4) — Percentage Distributions',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 🔍 Reflection Question 7

- Looking at the cluster profiles table and percentage plots, can you give each cluster a meaningful label? (e.g., "young lower-education workers", "high-capital older professionals")
- Which cluster has the highest proportion of >\$50K earners? What features distinguish it from the others?
- The percentage view (rather than raw counts) makes the age and education distributions of small and large clusters directly comparable. What would you have missed if you used raw counts?
- We clustered on only five continuous features. How might the clusters change if you added marital status or occupation? What would be the ethical implications of adding race or sex to the clustering features?

---
## Section 10: Reporting ML Results in Academic Writing

Machine learning findings need to be reported with the same rigour as any statistical analysis.
Below is a template for how you would report the results of this study in an academic journal,
followed by an APA-format results table.

---

### Example Methods Paragraph

> Three supervised classification models were compared in their ability to predict whether an individual's annual income exceeded \$50,000, using the 1994 US Census Income dataset (*N* = 48,842; Becker & Kohavi, 1996). The dataset comprised demographic and employment predictors including age, educational attainment, occupation, marital status, sex, race, hours worked per week, and capital gain and loss. Missing values (6.2% of cases across three variables: workclass, occupation, and native-country) were handled using iterative multiple imputation (van Buuren & Groothuis-Oudshoorn, 2011), implemented via scikit-learn's `IterativeImputer` (Pedregosa et al., 2011). Categorical predictors were dummy-coded prior to analysis. The dataset was partitioned into a training set (80%; *n* = 39,073) and a held-out test set (20%; *n* = 9,769), with stratified sampling used to preserve the natural class distribution (75.1% ≤\$50K; 24.9% >\$50K). Models evaluated included a classification decision tree (maximum depth = 5), a random forest (200 trees; maximum features = √*p*), and a gradient boosting classifier (200 estimators; learning rate = 0.10; maximum depth = 4; row subsampling fraction = 0.80). Model performance was assessed on the held-out test set using accuracy, precision, recall, and F1 score for the minority class (income >\$50K).

---

### Example Results Paragraph

> Table 1 presents the classification performance of all three models on the held-out test set. The gradient boosting classifier achieved the highest overall accuracy and F1 score for the minority class (>\$50K), followed closely by the random forest. The decision tree, whilst the most interpretable, showed a larger gap between training (XX%) and test accuracy (XX%), indicating modest overfitting at this depth. Across all models, precision and recall for the majority class (≤\$50K) consistently exceeded those for the minority class, reflecting the class imbalance in the original data. Variable importance analysis from the random forest model identified capital gain, education level, age, and marital status as the strongest predictors of high income, consistent with established findings on income stratification (Chetty et al., 2018). An unsupervised k-means cluster analysis (*k* = 4) on demographic and employment features identified four distinct socioeconomic profiles, which differed substantially in their proportion of high earners, suggesting meaningful latent heterogeneity within this population.

---

### APA-Format Table

In [ ]:
# ── Produce the APA-style results table ──────────────────────────────────────
# Fill in your actual values below from the results dataframe.

print("Table 1")
print("Classification Performance of Three ML Models on the Held-Out Test Set")
print()

header = f"{'Model':<22} {'Accuracy':>10} {'Precision':>11} {'Recall':>9} {'F1':>7}"
sep    = "-" * len(header)
print(header)
print(sep)

for _, row in results.iterrows():
    print(f"{row['Model']:<22} {row['Accuracy']:>10.3f} "
          f"{row['Precision (>50K)']:>11.3f} "
          f"{row['Recall (>50K)']:>9.3f} "
          f"{row['F1 (>50K)']:>7.3f}")

print()
print("Note. Metrics are reported on the held-out test set (20% of N = 48,842).")
print("Precision, Recall, and F1 are for the minority class (income >$50K).")
print("Models trained on a stratified 80% training partition.")
print("Baseline accuracy (majority-class classifier) = "
      f"{(1-y_test.mean())*100:.1f}%.")

### APA Citation for scikit-learn

When using scikit-learn in academic work, cite:

> Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., ... & Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, *12*, 2825–2830.

For multiple imputation (van Buuren & Groothuis-Oudshoorn):

> van Buuren, S., & Groothuis-Oudshoorn, K. (2011). mice: Multivariate imputation by chained equations in R. *Journal of Statistical Software*, *45*(3), 1–67.

---
## Wrap-Up

Well done for completing the practical.

In [ ]:
print("=" * 62)
print("WORKSHOP SUMMARY: Model Performance on Test Set")
print("=" * 62)
print(results.to_string(index=False))
print()
print(f"Baseline (majority-class): {(1-y_test.mean())*100:.1f}%  |  Recall >50K = 0.0%")
print()
print("Key takeaways:")
print("  1. Start with a decision tree: fast, interpretable, good baseline.")
print("     Use the depth-tuning plot to choose max_depth.")
print("  2. Random forests are robust and require minimal tuning.")
print("     Variable importance (MDI) helps you understand which features drive predictions.")
print("  3. Gradient boosting often achieves the best accuracy, but")
print("     requires more time and more careful hyperparameter choices.")
print("  4. Accuracy alone is misleading with imbalanced data.")
print("     Always check precision, recall, and F1 for the minority class.")
print("  5. ML finds patterns. It does not identify causes.")
print("     Use it alongside theory, not instead of it.")

---
## Where to Go Next

**Books**
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2023). *An Introduction to Statistical Learning* (2nd ed.). Free PDF at [statlearning.com](https://www.statlearning.com)
- Géron, A. (2022). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly.

**Free courses**
- [fast.ai](https://www.fast.ai) — practical ML, no maths prerequisites
- [Kaggle Learn](https://www.kaggle.com/learn) — short, hands-on ML tutorials

**Apply it to your research**
- What prediction task exists in your data?
- Which variables would you use as features?
- What would a model's predictions actually *be used for* in your field — and what risks would that introduce?

---
*Materials: https://github.com/BHannan01/introduction-to-machine-learning*
*Contact: brody.hannan@soton.ac.uk*